# Part 3: Feature Engineering 
## 3.1 Reproducible Pipeline Build a Preprocessing Pipeline


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import pandas as pd
from pathlib import Path

# Load Data
BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "processed" / "train_clean.csv"

df = pd.read_csv(DATA_PATH)

# Identify feature types for the pipeline
text_feature = 'review_text'
structured_features = [
    'product_price', 'product_category', 'seller_rating', 'delivery_days',
    'product_age_months', 'num_previous_reviews_by_user', 'verified_purchase',
    'return_initiated', 'helpful_votes', 'discount_pct', 'product_weight_kg', 'image_count'
]

# 1. Define the Text Pipeline with updated TF-IDF parameters
text_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2), min_df=5, max_df=0.9)),
    ('svd', TruncatedSVD(n_components=100, random_state=42))
])

# 2. Define the Structured Pipeline
structured_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

# 3. Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('text', text_pipeline, text_feature),
        ('struct', structured_pipeline, structured_features)
    ]
)

# Fit and transform the data
processed_features = preprocessor.fit_transform(df)

# Create a DataFrame for the combined matrix
text_names = [f'svd_comp_{i}' for i in range(100)]
struct_names = structured_features
combined_df = pd.DataFrame(processed_features, columns=text_names + struct_names)

print(f"Original shape: {df[structured_features + [text_feature]].shape}")
print(f"Processed feature matrix shape: {combined_df.shape}")
print("\n--- Combined Feature Matrix Preview ---")
display(combined_df.head())
print("\nPipeline successfully built and applied with updated TF-IDF parameters.")

Original shape: (2000, 13)
Processed feature matrix shape: (2000, 112)

--- Combined Feature Matrix Preview ---


,svd_comp_0,svd_comp_1,svd_comp_2,svd_comp_3,svd_comp_4,svd_comp_5,svd_comp_6,svd_comp_7,svd_comp_8,svd_comp_9,...,seller_rating,delivery_days,product_age_months,num_previous_reviews_by_user,verified_purchase,return_initiated,helpful_votes,discount_pct,product_weight_kg,image_count
0,0.403091,-0.318258,-0.095046,-0.105421,0.063587,-0.034929,0.017904,0.260654,0.066167,0.175566,...,-0.149005,-0.939529,1.687625,-0.589867,0.664020,-0.403473,0.712460,-0.863198,-0.815532,0.252794
1,0.326019,-0.209576,-0.227999,0.288466,0.023225,-0.001576,-0.062813,-0.057290,-0.044375,-0.129705,...,0.826639,-0.939529,-0.206222,1.144185,0.664020,-0.403473,0.000712,1.657604,-0.681947,0.252794
2,0.168141,-0.028121,0.037331,-0.009314,-0.014322,0.020253,-0.053896,-0.034158,-0.034288,-0.111027,...,-0.067702,-0.322534,0.859067,-0.589867,0.664020,-0.403473,-0.711037,-1.534950,-0.222746,0.687520
3,0.300123,-0.221181,-0.072585,0.109962,-0.080590,-0.018962,0.015754,-0.174273,-0.029883,-0.011567,...,0.257513,-0.322534,0.740701,-1.745901,0.664020,-0.403473,-0.711037,-0.212222,0.236455,0.252794
4,0.303906,-0.061133,-0.072833,0.155975,0.074752,0.011236,-0.139387,-0.065871,-0.169621,-0.137904,...,-0.523002,-0.939529,-0.738867,-0.589867,-1.505979,-0.403473,1.424209,0.618812,4.102091,-0.181933



Pipeline successfully built and applied with updated TF-IDF parameters.


#### Observations:
TF_IDF:
* max_features=5000, since based on EDA, the vocabulary is on the smaller side, it is only 324 words, which means that 5000 will help to ensure that all useful words and bigrams as well are captured (bigrams will incerease the vocabulary).

* ngram_range=(1,2) will help to add bigrams to understand the context and improve classification, since words like "not good" may often be in reviews.

* min_df=5 ensures that all very rare words are removed since they are noise, and since there are 2000 rows in this dataset, words that are there less than 5 times are not very significant, and it will be hard to generalize them.

* max_df=0.9, which means that it will remove very common words,since they are not informative and will not help in this case.

* stop_words='english' removes words that are noise and do not mean anything, like "and", "this", etc.

Truncated SVD:

* n_components=100, because the dataset is 2000 rows, and this number will help to avoid overfitting and reduce some noise. It will be helpful since TF-IDF makes it sparse and high-dimensional. If the number of components is lower, it may lose important information, but if it is a large number, then it will have too much noise.

StandardScaler will help to balance between features, since some of the has a big range and some do not, so it will standardize it.

Column Transofrmer needs to be used since the dataset contains text and structured features, so it will allow combining into a single feature matrix. It will also help during the training to avoid leakage.
